In [44]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [24]:
qld = pd.read_csv(r"C:\Users\Rudra\Desktop\kaggle\dsj\data\QL_data.csv")
ql_s = pd.read_csv(r"C:\Users\Rudra\Desktop\kaggle\dsj\data\QL_to_source.csv")
ql_s5 = pd.read_csv(r"C:\Users\Rudra\Desktop\kaggle\dsj\data\QL_to_source5.csv")

cd = pd.read_csv(r"C:\Users\Rudra\Desktop\kaggle\dsj\data\campaign_data.csv")
cs = pd.read_csv(r"C:\Users\Rudra\Desktop\kaggle\dsj\data\campaign_to_source.csv")
cs5 = pd.read_csv(r"C:\Users\Rudra\Desktop\kaggle\dsj\data\campaign_to_source5.csv")

In [25]:
qld.rename(columns={'lead id': 'lead_id'}, inplace=True)
qld.head(3)

,lead_id,campaign_code,lead_status,total_dial_count
0,lead_32078,dean,Intent Qualified,2
1,lead_32157,dean,Intent Qualified,3
2,lead_32175,dean,Intent Qualified,3


In [26]:
cs.sample(3)

,lead_id,source_code,category code
7734,lead_7735,source2,source2_4
18156,lead_18172,source6,NaN
54219,lead_54551,source1,source1_157


## Total Leads

In [27]:
total_lead_per_source_df = pd.merge(cs, qld, on="lead_id",  how='left')
total_lead_per_source_df.head(3)

,lead_id,source_code,category code,campaign_code,lead_status,total_dial_count
0,lead_1,source1,source1_1,NaN,NaN,NaN
1,lead_2,source5,NaN,multi,Intent Qualified,1.0
2,lead_3,source1,source1_2,NaN,NaN,NaN


In [34]:
total_leads = total_lead_per_source_df.groupby("source_code")["lead_id"].nunique().reset_index()
total_leads.rename(columns={"lead_id": "total_leads"}, inplace=True)
total_leads

,source_code,total_leads
0,source 5,77
1,source1,26309
2,source2,12524
3,source2,179
4,source3,4185
5,source4,2676
6,source5,9971
7,source5,150
8,source6,2134
9,source7,4442


## Qualified leads per source

In [28]:
qualified_leads_per_source_df = pd.merge(qld, ql_s, on='lead_id', how='left')
qualified_leads_per_source_df.head(3)

,lead_id,campaign_code,lead_status,total_dial_count,source_code,category code
0,lead_32078,dean,Intent Qualified,2,source2,source2_3
1,lead_32157,dean,Intent Qualified,3,source2,source2_4
2,lead_32175,dean,Intent Qualified,3,source1,source1_7


In [36]:
qualified_leads = qualified_leads_per_source_df.groupby("source_code")["lead_id"].nunique().reset_index()
qualified_leads.rename(columns={"lead_id": "qualified_leads"}, inplace=True)
qualified_leads

,source_code,qualified_leads
0,source1,214
1,source2,294
2,source3,26
3,source4,14
4,source5,82
5,source6,5
6,source7,12


In [ ]:
conversion_df = total_leads.merge(qualified_leads, on="source_code", how="left")

conversion_df['qualified_leads'] = conversion_df['qualified_leads'].fillna(0)

conversion_df['conversion_rate'] = np.round( (conversion_df['qualified_leads'] / conversion_df['total_leads'] ) * 1000, 2)

conversion_df.sort_values(by="conversion_rate", ascending=True)

conversion_df["share_of_total"] = conversion_df['total_leads'] / conversion_df['total_leads'].sum()

conversion_df

,source_code,total_leads,qualified_leads,conversion_rate,share_of_total
0,source 5,77,0.0,0.00,0.001229
1,source1,26309,214.0,8.13,0.419956
2,source2,12524,294.0,23.47,0.199914
3,source2,179,0.0,0.00,0.002857
4,source3,4185,26.0,6.21,0.066803
5,source4,2676,14.0,5.23,0.042716
6,source5,9971,82.0,8.22,0.159162
7,source5,150,0.0,0.00,0.002394
8,source6,2134,5.0,2.34,0.034064
9,source7,4442,12.0,2.70,0.070905


# source wise campaigns

In [48]:
df = cd.merge(cs, on="lead_id", how="left")
ql_df = qld.merge(ql_s, on="lead_id", how="left")
ql_df

,lead_id,campaign_code,lead_status,total_dial_count,source_code,category code
0,lead_32078,dean,Intent Qualified,2,source2,source2_3
1,lead_32157,dean,Intent Qualified,3,source2,source2_4
2,lead_32175,dean,Intent Qualified,3,source1,source1_7
3,lead_7021,asset,Intent Qualified,1,source3,source3_6
4,lead_59345,foli,Intent Qualified,2,source2,source2_4
...,...,...,...,...,...,...
664,lead_59316,foli,Intent Qualified,9,source4,source4_1
665,lead_31998,dean,Intent Qualified,1,source2,source2_4
666,lead_45602,multi,Qualified,2,source2,source2_4
667,lead_45485,multi,Intent Qualified,5,source2,source2_3


In [49]:
total = df.groupby(["source_code", "campaign_code"])["lead_id"].nunique().reset_index()
total.rename(columns={"lead_id": "total_leads"}, inplace=True)
total

,source_code,campaign_code,total_leads
0,source 5,asset,27
1,source 5,dean,25
2,source 5,foli,25
3,source1,asset,5825
4,source1,dean,11755
5,source1,foli,5482
6,source1,multi,3247
7,source2,asset,652
8,source2,dean,5974
9,source2,foli,1083


In [50]:
qualified = ql_df.groupby(["source_code", "campaign_code"])["lead_id"].nunique().reset_index()
qualified.rename(columns={"lead_id": "qualified_leads"}, inplace=True)

In [52]:
final = total.merge(qualified, on=["source_code", "campaign_code"], how="left")

final["qualified_leads"] = final["qualified_leads"].fillna(0)
final["conversion_rate"] = final["qualified_leads"] / final["total_leads"]

final = final.sort_values(["source_code", "conversion_rate"], ascending=[True, False])

final

,source_code,campaign_code,total_leads,qualified_leads,conversion_rate
0,source 5,asset,27,0.0,0.000000
1,source 5,dean,25,0.0,0.000000
2,source 5,foli,25,0.0,0.000000
6,source1,multi,3247,47.0,0.014475
4,source1,dean,11755,116.0,0.009868
3,source1,asset,5825,39.0,0.006695
5,source1,foli,5482,12.0,0.002189
8,source2,dean,5974,160.0,0.026783
9,source2,foli,1083,26.0,0.024007
10,source2,multi,4815,99.0,0.020561
